In [5]:
%%capture
!pip install open-clip-torch kagglehub albumentations scikit-learn seaborn


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import glob
from PIL import Image
from tqdm.notebook import tqdm
import random

# OpenCLIP for SigLIP model
import open_clip

# Image transformations
import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

# Metrics and plotting
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Google Drive mounting and JSON saving
from google.colab import drive
import json


## 2. Configuration

In [7]:
BACKBONE_NAME = "ViT-B-16-SigLIP-384"
PRETRAINED    = "webli"
RESOLUTION    = 384
N_SHOT        = 50          # support examples per class
MAX_TOTAL_IMAGES_PER_DATASET = 5000 # Limit total images to 1000 per dataset (500 per class for balanced sampling)
SEED          = 42
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "/content/drive/MyDrive/deepfake_assignment/results/best_Hidf.pt"
TAU           = 1.0

print(f"Using device: {DEVICE}")
print(f"Model backbone: {BACKBONE_NAME}")
print(f"N-shot: {N_SHOT}")
print(f"Resolution: {RESOLUTION}")
print(f"Max total images per dataset: {MAX_TOTAL_IMAGES_PER_DATASET}")

Using device: cuda
Model backbone: ViT-B-16-SigLIP-384
N-shot: 50
Resolution: 384
Max total images per dataset: 5000


## 3. Model Definition

We define the `SigLIP2Classifier` class. This matches the architecture used during HiDF training, allowing us to load the checkpoint correctly. The `encode` method will extract L2-normalized features directly from the frozen backbone, discarding the classification head.

In [8]:
class SigLIP2Classifier(nn.Module):
    def __init__(self, backbone_name, pretrained, embed_dim=768, hidden_dim=256, dropout_rate=0.1):
        super().__init__()
        self.backbone, _, _ = open_clip.create_model_and_transforms(
            backbone_name, pretrained=pretrained,
            device="cpu" # Load to CPU first to avoid OOM if loading multiple models
        )

        # Freeze backbone parameters
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Classification head (matching the paper's description for HiDF training)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, 1) # Single output for binary classification
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone.encode_image(x)
            features = F.normalize(features, dim=-1) # L2 normalize backbone features
        return self.head(features)

    @torch.no_grad()
    def encode(self, images):
        """
        Extracts L2-normalized embeddings using only the frozen backbone.
        """
        # Ensure backbone is in eval mode
        self.backbone.eval()
        embeddings = self.backbone.encode_image(images)
        # L2-normalize the embeddings as per Equation (2)
        embeddings = F.normalize(embeddings.float(), dim=-1)
        return embeddings


## 4. Load checkpoint & extract encoder

In [9]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Instantiate the model architecture
model = SigLIP2Classifier(BACKBONE_NAME, PRETRAINED)

# Load the checkpoint weights
if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

    # Filter checkpoint state_dict to handle specific issues:
    # 1. Skip keys related to the VisionTransformer's internal head (if present)
    # 2. Skip 'pos_embed' if its size does not match the currently initialized model's pos_embed.
    state_dict_to_load = {}
    model_visual_state_dict = model.backbone.visual.state_dict() # Get current model's visual state_dict

    for k, v in checkpoint['model_state_dict'].items():
        # Skip keys related to the VisionTransformer's internal classification head
        if k.startswith('head.'):
            print(f"Skipping checkpoint key: '{k}' (belongs to VisionTransformer's internal head, not used by SigLIP2Classifier's encoder)")
            continue

        # Handle pos_embed size mismatch: prefer current model's pos_embed if sizes differ
        if k == 'pos_embed':
            if k in model_visual_state_dict and v.shape != model_visual_state_dict[k].shape:
                print(f"Skipping checkpoint key: '{k}' due to size mismatch. "
                      f"Checkpoint shape {v.shape}, current model shape {model_visual_state_dict[k].shape}. "
                      f"Using current model's '{k}'.")
                continue

        state_dict_to_load[k] = v

    try:
        # Load the filtered state_dict into model.backbone.visual (the VisionTransformer module).
        # Using strict=False to allow for any remaining minor discrepancies or missing keys
        # after manual filtering. Keys not in state_dict_to_load will remain as initialized.
        model.backbone.visual.load_state_dict(state_dict_to_load, strict=False)
        print("Successfully loaded filtered weights into model.backbone.visual (VisionTransformer).")

    except RuntimeError as e:
        print(f"Error loading filtered checkpoint into model.backbone.visual: {e}")
        raise ValueError(f"Checkpoint loading failed after filtering. Details: {e}")
else:
    raise FileNotFoundError(f"Checkpoint not found at {CHECKPOINT_PATH}. Please ensure it's in your Google Drive.")

# Move model to the specified device
model.to(DEVICE)

# Set model to evaluation mode and freeze all parameters (already done in __init__ but good practice)
model.eval()
for param in model.parameters():
    param.requires_grad = False

print("Model loaded and set to evaluation mode on device:", DEVICE)

## 5. Download Dataset 1: AI vs Human

In [ ]:
import kagglehub

def parse_dataset_structure(root_path, label_map):
    """
    Walks through the dataset path to find image files and their labels.
    Assumes direct subfolders of root_path are class folders.
    """
    all_images = []
    all_labels = []
    class_counts = {class_name: 0 for class_name in label_map.keys()}

    # Iterate through all subdirectories in the root_path
    for class_folder_name, label_value in label_map.items():
        class_path = os.path.join(root_path, class_folder_name)
        if not os.path.exists(class_path):
            print(f"Warning: Class folder '{class_folder_name}' not found at {class_path}")
            continue

        for dirpath, dirnames, filenames in os.walk(class_path):
            for filename in filenames:
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(dirpath, filename)
                    all_images.append(img_path)
                    all_labels.append(label_value)
                    class_counts[class_folder_name] += 1
    return all_images, all_labels, class_counts

def parse_dataset_from_dataframe(dataset_base_path, dataframe, filename_col='file_name', label_col='label'):
    """
    Parses dataset information (image paths and labels) from a pandas DataFrame.
    Assumes 'filename_col' contains relative paths and 'label_col' contains integer labels.
    """
    all_images = []
    all_labels = []
    unique_labels = sorted(dataframe[label_col].unique())
    class_counts = {label: 0 for label in unique_labels} # Use integer labels for counts

    for idx, row in dataframe.iterrows():
        relative_path = row[filename_col]
        # Construct the full path by joining the base dataset path with the relative path from the dataframe
        img_path = os.path.join(dataset_base_path, relative_path)
        if os.path.exists(img_path):
            all_images.append(img_path)
            label = row[label_col]
            all_labels.append(label)
            class_counts[label] += 1
        # else:
        #     print(f"Warning: Image not found at {img_path}") # Uncomment for debugging missing files
    return all_images, all_labels, class_counts

# Download Dataset 1: AI vs Human
ai_human_path = kagglehub.dataset_download("alessandrasala79/ai-vs-human-generated-dataset")
print(f"Dataset 1 downloaded to: {ai_human_path}")

# Load the train.csv for AI vs Human dataset (ai_human_df is already in kernel state)
train_csv_path_ai_human = os.path.join(ai_human_path, "train.csv")
ai_human_df = pd.read_csv(train_csv_path_ai_human)

# Using the new dataframe parser for Dataset 1
ai_human_images, ai_human_labels, ai_human_class_counts = parse_dataset_from_dataframe(ai_human_path, ai_human_df)

# Adjust label_map for display (mapping integer labels to names)
ai_human_label_map_reversed = {v: k for k, v in {"AI": 1, "human": 0}.items()}

print("\nDataset 1 (AI vs Human) Class Counts:")
for class_label, count in ai_human_class_counts.items():
    class_name = ai_human_label_map_reversed.get(class_label, f"Class {class_label}")
    print(f"  {class_name}: {count} images")
print(f"Total images in Dataset 1: {len(ai_human_images)}")

## 6. Download Dataset 2: Deepfake vs Real 60K

In [ ]:
# Download Dataset 2: Deepfake vs Real 60K
deepfake_path = kagglehub.dataset_download("prithivsakthiur/deepfake-vs-real-60k")
print(f"Dataset 2 downloaded to: {deepfake_path}")

# Adjust label_map based on actual folder names after extraction
deepfake_label_map = {"Fake": 1, "Real": 0} # Corrected to use capitalized folder names

# Corrected path for Dataset 2, using the folder-based parser
deepfake_dataset_root = os.path.join(deepfake_path, "deepfake-vs-real-60k")
deepfake_images, deepfake_labels, deepfake_class_counts = parse_dataset_structure(deepfake_dataset_root, deepfake_label_map)

print("\nDataset 2 (Deepfake vs Real 60K) Class Counts:")
for class_name, count in deepfake_class_counts.items():
    print(f"  {class_name}: {count} images")
print(f"Total images in Dataset 2: {len(deepfake_images)}")

The previous `parse_dataset_structure` calls still show warnings, indicating that the class folders (e.g., 'AI', 'human') are not directly under the paths `ai_human_dataset_root` and `deepfake_dataset_root` as expected. Let's inspect the contents of these *corrected* root paths to confirm their structure.

### Inspect `train.csv` for AI vs Human Dataset

Since the `ai_human_dataset_root/train_data` directory contains images directly and not in class subfolders, we need to inspect the `train.csv` file, which is located in the `ai_human_path` root. This CSV likely contains the image filenames and their corresponding labels.

In [ ]:
train_csv_path_ai_human = os.path.join(ai_human_path, "train.csv")
ai_human_df = pd.read_csv(train_csv_path_ai_human)
print(f"Contents of {train_csv_path_ai_human}:")
display(ai_human_df.head())

### Inspecting Dataset Folder Structures

It appears the `parse_dataset_structure` function could not find any images. This often happens if the downloaded datasets are extracted into an additional subfolder within the main dataset directory. Let's inspect the actual folder structure to correct the paths.

In [ ]:
print(f"Contents of Dataset 1 root ({ai_human_path}):")
!ls -F {ai_human_path}

print(f"\nContents of Dataset 2 root ({deepfake_path}):")
!ls -F {deepfake_path}

## 7. Few-Shot Adaptation Function

In [ ]:
class FewShotDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=np.array(image))["image"]

        return image, label, img_path # Return image_path for tracking

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@torch.no_grad()
def extract_embeddings(model, dataloader, device, use_tta=False):
    embeddings = []
    labels = []
    image_paths_list = []
    model.eval()

    # Define TTA transform if needed
    # Note: the base transform (resize, ToTensor, normalize) is applied by the dataloader's transform
    # This TTA only applies horizontal flip on top of that.
    tta_transform = A.Compose([
        A.HorizontalFlip(p=1.0),
        # Assuming the original transform already includes Resize, ToTensorV2, Normalize
        # If not, add them here explicitly based on your base_transform
    ])

    with torch.amp.autocast(device_type=device):
        for images, batch_labels, batch_paths in tqdm(dataloader, desc="Extracting Embeddings" + (" (with TTA)" if use_tta else "")):
            images = images.to(device)

            # Original image embeddings
            original_embeddings = model.encode(images)

            if use_tta:
                # Apply TTA (horizontal flip) and get embeddings for flipped images
                # For TTA, we need to convert tensor to PIL Image, flip, then reapply transform.
                # This requires torchvision.transforms.functional as TF
                import torchvision.transforms.functional as TF # Import locally for this scope
                from torchvision.transforms import ToPILImage

                flipped_images = []
                for img_tensor in images:
                    # Convert tensor to PIL Image (undo normalization if any, for visual flip)
                    img_pil = ToPILImage()(img_tensor.cpu())
                    img_flipped_pil = TF.hflip(img_pil)
                    # Reapply the same test_transform (resize, normalize, to_tensor) to the flipped image
                    flipped_images.append(dataloader.dataset.transform(image=np.array(img_flipped_pil))['image'])
                flipped_images = torch.stack(flipped_images).to(device)
                flipped_embeddings = model.encode(flipped_images)
                # Average the embeddings of original and flipped images
                final_batch_embeddings = (original_embeddings + flipped_embeddings) / 2
            else:
                final_batch_embeddings = original_embeddings

            embeddings.append(final_batch_embeddings.cpu())
            labels.append(batch_labels.cpu())
            image_paths_list.extend(batch_paths)

    return torch.cat(embeddings), torch.cat(labels), image_paths_list


def run_few_shot_adaptation(
    model,           # loaded model with encode() method
    dataset_loader_config, # dict containing dataset loading info
    dataset_name,    # string for logging
    n_shot=50,
    tau=1.0,
    seed=42,
    resolution=384,
    batch_size=32,
    device=DEVICE,
    max_total_images_per_dataset=None # New parameter for limiting total images
):
    set_seed(seed)
    print(f"\n--- Running Few-Shot Adaptation for {dataset_name} ---")

    # 1. Image Transformations
    # Base transform for inference (resize, ToTensor, Normalize)
    test_transform = A.Compose([
        A.Resize(resolution, resolution),
        A.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)),
        ToTensorV2()
    ])

    # 2. Load ALL images from dataset
    if dataset_loader_config['type'] == 'folder_structure':
        all_image_paths, all_labels, _ = parse_dataset_structure(
            dataset_loader_config['root_path'],
            dataset_loader_config['label_map']
        )
    elif dataset_loader_config['type'] == 'dataframe_csv':
        all_image_paths, all_labels, _ = parse_dataset_from_dataframe(
            dataset_loader_config['base_path'],
            dataset_loader_config['dataframe'],
            dataset_loader_config['filename_col'],
            dataset_loader_config['label_col']
        )
    else:
        raise ValueError("Invalid dataset_loader_config type specified.")

    print(f"Found {len(all_image_paths)} total images for {dataset_name}.")

    # Group images by class for stratified sampling
    class_to_images = {0: [], 1: []}
    for img_path, label in zip(all_image_paths, all_labels):
        class_to_images[label].append(img_path)

    # Limit total images if specified, maintaining class balance
    if max_total_images_per_dataset is not None:
        limited_image_paths = []
        limited_labels = []
        max_per_class = max_total_images_per_dataset // len(class_to_images) # Assuming 2 classes, so 500 per class
        print(f"Limiting dataset to {max_total_images_per_dataset} images ({max_per_class} per class).")
        for class_label in sorted(class_to_images.keys()):
            random.shuffle(class_to_images[class_label]) # Shuffle before taking subset
            limited_image_paths.extend(class_to_images[class_label][:max_per_class])
            limited_labels.extend([class_label] * min(len(class_to_images[class_label]), max_per_class))

        # Re-assign to use the limited set
        all_image_paths = limited_image_paths
        all_labels = limited_labels

        # Re-group after limiting
        class_to_images = {0: [], 1: []}
        for img_path, label in zip(all_image_paths, all_labels):
            class_to_images[label].append(img_path)
        print(f"After limiting, found {len(all_image_paths)} total images for {dataset_name}.")


    # 3. Randomly sample n_shot per class for support set
    support_image_paths = []
    support_labels = []
    query_image_paths = []
    query_labels = []

    for class_label in sorted(class_to_images.keys()):
        class_images = class_to_images[class_label]
        random.shuffle(class_images) # Shuffle for random sampling

        if len(class_images) < n_shot:
            print(f"Warning: Not enough images ({len(class_images)}) for class {class_label} to sample {n_shot} shots. Using all available for support.")
            support_image_paths.extend(class_images)
            support_labels.extend([class_label] * len(class_images))
        else:
            support_image_paths.extend(class_images[:n_shot])
            support_labels.extend([class_label] * n_shot)
            query_image_paths.extend(class_images[n_shot:])
            query_labels.extend([class_label] * len(class_images[n_shot:]))

    print(f"Support set: {len(support_image_paths)} images ({n_shot} per class).")
    print(f"Query set: {len(query_image_paths)} images.")

    # Create datasets and dataloaders
    support_dataset = FewShotDataset(support_image_paths, support_labels, test_transform)
    query_dataset = FewShotDataset(query_image_paths, query_labels, test_transform) # TTA handled during extraction

    support_dataloader = torch.utils.data.DataLoader(support_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    query_dataloader = torch.utils.data.DataLoader(query_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # 4. Extract embeddings for support set
    support_embeddings, support_labels_tensor, _ = extract_embeddings(model, support_dataloader, device, use_tta=False)

    # 5. Compute & L2-normalize prototypes
    prototypes = []
    for class_label in sorted(list(set(support_labels_tensor.tolist()))):
        class_embeddings = support_embeddings[support_labels_tensor == class_label]
        prototype = F.normalize(class_embeddings.mean(dim=0), dim=-1) # Mean and L2-normalize
        prototypes.append(prototype)
    prototypes = torch.stack(prototypes).to(device) # Move prototypes to device for similarity computation

    print(f"Computed {len(prototypes)} prototypes.")

    # 6. Extract embeddings for query set (in batches, with TTA)
    query_embeddings, true_labels_tensor, _ = extract_embeddings(model, query_dataloader, device, use_tta=True) # TTA applied here
    query_embeddings = query_embeddings.to(device) # Move query embeddings to device

    # 7. Query Classification - Equation (4)
    # Compute L2 distance to prototypes
    # ||z(x) - p_c||₂^2 = ||z(x)||^2 + ||p_c||^2 - 2 * z(x) . p_c
    # Since z(x) and p_c are L2-normalized, ||z(x)||^2 = 1 and ||p_c||^2 = 1
    # So, ||z(x) - p_c||₂^2 = 2 - 2 * z(x) . p_c
    # Distance = sqrt(2 - 2 * z(x) . p_c)

    # We are interested in -Distance / tau, so -sqrt(2 - 2 * z(x) . p_c) / tau
    # Cosine similarity is z(x) . p_c. Higher similarity means lower distance.
    # So, we can just use similarity directly in softmax by negating it (or negating distance)

    # The paper uses -||z(x) - p_c||₂ / τ. Let's calculate Euclidean distance.

    distances = torch.cdist(query_embeddings, prototypes, p=2) # Euclidean distance

    # Compute P(y=c|x) using softmax over negative distances
    logits = -distances / tau
    probabilities = F.softmax(logits, dim=1)

    # For binary classification (two classes), we need the probability of class 1 (fake)
    # Assuming prototypes[0] is for class 0 (real) and prototypes[1] is for class 1 (fake)
    # So, probabilities[:, 1] is P(y=1|x)
    fake_probabilities = probabilities[:, 1].cpu().numpy()

    # 8. Apply threshold (0.5 default)
    predictions = (fake_probabilities >= 0.5).astype(int)

    # 9. Evaluation Metrics
    true_labels = true_labels_tensor.numpy()

    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, average='macro', zero_division=0)
    recall = recall_score(true_labels, predictions, average='macro', zero_division=0)
    f1 = f1_score(true_labels, predictions, average='macro', zero_division=0)
    roc_auc = roc_auc_score(true_labels, fake_probabilities)
    avg_precision = average_precision_score(true_labels, fake_probabilities)
    total_samples = len(true_labels)
    cm = confusion_matrix(true_labels, predictions)

    # ROC Curve data
    fpr, tpr, roc_thresholds = roc_curve(true_labels, fake_probabilities)

    # Precision-Recall Curve data
    prec, rec, pr_thresholds = precision_recall_curve(true_labels, fake_probabilities)

    metrics = {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "average_precision": avg_precision,
        "total_samples": total_samples,
        "confusion_matrix": cm.tolist(), # Convert to list for JSON serialization
        "roc_curve": {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "thresholds": roc_thresholds.tolist()},
        "pr_curve": {"precision": prec.tolist(), "recall": rec.tolist(), "thresholds": pr_thresholds.tolist()}
    }

    print(f"Evaluation results for {dataset_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision (macro): {precision:.4f}")
    print(f"  Recall (macro): {recall:.4f}")
    print(f"  F1 (macro): {f1:.4f}")
    print(f"  AUC-ROC: {roc_auc:.4f}")
    print(f"  Average Precision: {avg_precision:.4f}")
    print(f"  Total Samples: {total_samples}")

    return {
        "true_labels": true_labels,
        "probabilities": fake_probabilities,
        "predictions": predictions,
        "metrics": metrics
    }

## 8. Run on Dataset 1 (AI vs Human)

In [ ]:
import torch
ai_human_loader_config = {
    'type': 'dataframe_csv',
    'base_path': ai_human_path,
    'dataframe': ai_human_df,
    'filename_col': 'file_name',
    'label_col': 'label'
}

results_1 = run_few_shot_adaptation(
    model,
    ai_human_loader_config,
    dataset_name="AI vs Human Generated Images",
    n_shot=N_SHOT,
    tau=TAU,
    seed=SEED,
    resolution=RESOLUTION,
    batch_size=32,
    device=DEVICE,
    max_total_images_per_dataset=MAX_TOTAL_IMAGES_PER_DATASET
)

torch.summary()

## 9. Run on Dataset 2 (Deepfake vs Real 60K)

In [ ]:
deepfake_loader_config = {
    'type': 'folder_structure',
    'root_path': deepfake_dataset_root,
    'label_map': deepfake_label_map
}

results_2 = run_few_shot_adaptation(
    model,
    deepfake_loader_config,
    dataset_name="Deepfake vs Real 60K",
    n_shot=N_SHOT,
    tau=TAU,
    seed=SEED,
    resolution=RESOLUTION,
    batch_size=32,
    device=DEVICE,
    max_total_images_per_dataset=MAX_TOTAL_IMAGES_PER_DATASET
)

## 10. Comparison Table

In [ ]:
print("Comparison Table (Paper vs Our Results):\n")

# Expected paper results (for comparison)
paper_results = {
    "AI vs Human": {"accuracy": 0.8917, "precision": 0.8923, "recall": 0.8917, "f1": 0.8917, "total": 79850},
    "Deepfake-vs-Real": {"accuracy": 0.9820, "precision": 0.9819, "recall": 0.9819, "f1": 0.9820, "total": 91234}
}

# Prepare data for printing
table_header = "Dataset            | Accuracy | Precision | Recall | F1    | Total"
table_separator = "-------------------|----------|-----------|--------|-------|-------"

print(table_header)
print(table_separator)

# Print paper results
for dataset_name, metrics in paper_results.items():
    print(f"{dataset_name:<19}| {metrics['accuracy']:.4f}   | {metrics['precision']:.4f}    | {metrics['recall']:.4f} | {metrics['f1']:.4f}| {metrics['total']}")

# Print a separator for clarity between paper and our results
print(table_separator)

# Our results for Dataset 1
metrics_1 = results_1['metrics']
print(f"Our AI vs Human    | {metrics_1['accuracy']:.4f}   | {metrics_1['precision']:.4f}    | {metrics_1['recall']:.4f} | {metrics_1['f1']:.4f}| {metrics_1['total_samples']}")

# Our results for Dataset 2
metrics_2 = results_2['metrics']
print(f"Our Deepfake-Real  | {metrics_2['accuracy']:.4f}   | {metrics_2['precision']:.4f}    | {metrics_2['recall']:.4f} | {metrics_2['f1']:.4f}| {metrics_2['total_samples']}")


## 11. Visualizations

In [ ]:
def plot_metrics(results, dataset_name, output_dir="/content/drive/MyDrive/deepfake_assignment/Hidf_fewshots"):
    """
    Plots confusion matrix, ROC curve, and Precision-Recall curve for a given dataset.
    """
    metrics = results['metrics']
    true_labels = results['true_labels']
    probabilities = results['probabilities']
    predictions = results['predictions']

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'Evaluation for {dataset_name}', fontsize=16)

    # Plot 1: Confusion Matrix
    sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Real (0)', 'Fake (1)'], yticklabels=['Real (0)', 'Fake (1)'])
    axes[0].set_title('Confusion Matrix')
    axes[0].set_xlabel('Predicted Label')
    axes[0].set_ylabel('True Label')

    # Plot 2: ROC Curve
    fpr = metrics['roc_curve']['fpr']
    tpr = metrics['roc_curve']['tpr']
    roc_auc = metrics['roc_auc']
    axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('Receiver Operating Characteristic')
    axes[1].legend(loc="lower right")
    axes[1].grid(True)

    # Plot 3: Precision-Recall Curve
    prec = metrics['pr_curve']['precision']
    rec = metrics['pr_curve']['recall']
    avg_precision = metrics['average_precision']
    axes[2].plot(rec, prec, color='blue', lw=2, label=f'Precision-Recall curve (area = {avg_precision:.2f})')
    axes[2].set_xlim([0.0, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('Recall')
    axes[2].set_ylabel('Precision')
    axes[2].set_title('Precision-Recall Curve')
    axes[2].legend(loc="lower left")
    axes[2].grid(True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    filepath = os.path.join(output_dir, f"{dataset_name.replace(' ', '_').lower()}_metrics.png")
    plt.savefig(filepath)
    plt.show()
    print(f"Saved metrics plot for {dataset_name} to {filepath}")

# Plot for Dataset 1
plot_metrics(results_1, "AI vs Human Generated Images")

# Plot for Dataset 2
plot_metrics(results_2, "Deepfake vs Real 60K")


## 12. Save results to JSON

In [ ]:
output_json_path = "/content/drive/MyDrive/deepfake_assignment/Hidf_fewshots/fewshot_results.json"

# Ensure the directory exists
output_dir = os.path.dirname(output_json_path)
os.makedirs(output_dir, exist_ok=True)

# Collect all relevant data
full_results = {
    "config": {
        "BACKBONE_NAME": BACKBONE_NAME,
        "PRETRAINED": PRETRAINED,
        "RESOLUTION": RESOLUTION,
        "N_SHOT": N_SHOT,
        "SEED": SEED,
        "DEVICE": DEVICE,
        "CHECKPOINT_PATH": CHECKPOINT_PATH,
        "TAU": TAU
    },
    "dataset_1": {
        "name": "AI vs Human Generated Images",
        "results": results_1['metrics'],
        "true_labels": results_1['true_labels'].tolist(), # Convert numpy array to list for JSON
        "probabilities": results_1['probabilities'].tolist(),
        "predictions": results_1['predictions'].tolist()
    },
    "dataset_2": {
        "name": "Deepfake vs Real 60K",
        "results": results_2['metrics'],
        "true_labels": results_2['true_labels'].tolist(), # Convert numpy array to list for JSON
        "probabilities": results_2['probabilities'].tolist(),
        "predictions": results_2['predictions'].tolist()
    }
}

# Save to JSON
with open(output_json_path, 'w') as f:
    json.dump(full_results, f, indent=4)

print(f"All results saved to {output_json_path}")

In [ ]:
import torch
torch.summary()